Archives from DepMap were download from:
- DepMap Public 24Q4 Files [https://depmap.org/portal/data_page/?tab=allData]
    - Model.csv: https://depmap.org/portal/data_page/?tab=allData&releasename=DepMap%20Public%2024Q4&filename=Model.csv
    - OmicsDefaultModelProfiles.csv: https://depmap.org/portal/data_page/?tab=allData&releasename=DepMap%20Public%2024Q4&filename=OmicsDefaultModelProfiles.csv
    - OmicsProfiles.csv : https://depmap.org/portal/data_page/?tab=allData&releasename=DepMap%20Public%2024Q4&filename=OmicsProfiles.csv
    - OmicsExpressionProteinCodingGenesTPMLogp1.csv : https://depmap.org/portal/data_page/?tab=allData&releasename=DepMap%20Public%2024Q4&filename=OmicsExpressionProteinCodingGenesTPMLogp1.csv
    - OmicsSomaticMutations.csv : https://depmap.org/portal/data_page/?tab=allData&releasename=DepMap%20Public%2024Q4&filename=OmicsSomaticMutations.csv

Archive from GDSC were download from:
- https://cellmodelpassports.sanger.ac.uk/downloads
    - Drug Sensitivity Data [https://cmp.cog.sanger.ac.uk/download/GDSC2_fitted_dose_response_27Oct23.xlsx]

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
RAW_DIR = PROJECT_ROOT / "data" / "raw"

raw_files = [
    path for path in RAW_DIR.rglob("*")
    if path.is_file() and path.name != ".gitkeep"
]

records = []

for path in raw_files:
    sha256 = hashlib.sha256()
    
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)
    
    records.append({
        "relative_path": str(path.relative_to(PROJECT_ROOT)).replace("\\", "/"),
        "file_name": path.name,
        "dataset": path.parent.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256.hexdigest(),
    })

raw_file_audit = (
    pd.DataFrame(records)
    .sort_values(["dataset", "file_name"])
    .reset_index(drop=True)
)

raw_file_audit

In [ ]:
print(raw_file_audit)

In [ ]:
AUDIT_DIR = PROJECT_ROOT / "data" / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

raw_file_audit.to_json(
    AUDIT_DIR / "raw_file_audit.json",
    orient="records",
    indent=2
)